[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/03_real_world_io/A1_web_scraping_firecrawl.ipynb)

> 📎 **Appendix notebook — reference style.** This is one of the optional appendices (see `README.md`). Unlike the main course notebooks, appendices are written as a demo / reference: they focus on *seeing* a tool at work rather than on interactive exercises. The hands-on cells run on the **core course stack** (`requests`, `beautifulsoup4`, plus the standard library) against a **local HTML fixture** — so everything works offline with no network and no API key. The **Firecrawl** cells fall back to a small built-in mock; install `firecrawl-py` and set a key to swap in the real service.

---
# 📓 Notebook A1 (I/O) — Web Scraping & Firecrawl

> **Module:** Real-world I/O · **Type:** Appendix · **Estimated time:** 50–70 min · **Difficulty:** Intermediate

Notebook 12 pulled data from a **clean JSON API**. But most of the web has no API — the data you want is trapped in HTML meant for human eyes. **Web scraping** is the craft of extracting it anyway: politely, legally, and without your pipeline breaking every time a designer moves a `<div>`.

This appendix has two halves:

1. **Do it yourself** — `requests` + **BeautifulSoup**, the rules of the road (`robots.txt`, rate limits, the law), and where hand-rolled scraping starts to hurt.
2. **Let a service do the hard part** — **[Firecrawl](https://firecrawl.dev)**, an API that turns *any* URL (including JavaScript-heavy, anti-bot pages) into **clean, LLM-ready markdown** or **structured JSON** in one call — the modern way to feed a RAG pipeline or an agent.

---

## 🎯 Learning objectives
- Decide **whether** to scrape — and scrape **within the rules** (`robots.txt`, ToS, rate limits, GDPR/PII).
- Extract structured data from HTML with **BeautifulSoup** selectors, and write a **polite** fetcher (User-Agent, delay, cache, retries).
- Know the failure modes of DIY scraping (JS rendering, anti-bot, layout drift) — and when to reach for a managed service.
- Use **Firecrawl** to scrape / crawl / map / extract, and pipe the result into a validated, RAG-ready dataset.

## ✅ Prerequisites
- **Notebook 12 (APIs & HTTP)** — timeouts, status codes, retry/backoff (we build on it).
- Notebook 13 (SQL & validation) helps for the Pydantic step.

## 📦 Install

```bash
pip install requests beautifulsoup4     # the DIY stack (beautifulsoup4 imports as `bs4`)
pip install firecrawl-py                 # the Firecrawl Python SDK (imports as `firecrawl`)
# pip install playwright                 # only if you must render JavaScript yourself
```

## 1. When to scrape — and the rules you don't break

Scraping is a **last resort**, not a first move. Walk down this list and stop at the first "yes":

1. **Is there an official API or data export?** → Use it. (Notebook 12.) It's stabler, faster, and sanctioned.
2. **Is there a bulk dataset / RSS feed / sitemap?** → Use that.
3. **Only then** consider scraping the rendered page.

And once you scrape, you are a guest on someone else's server. The rules:

| Rule | What it means in practice |
|---|---|
| **Respect `robots.txt`** | Check `site.com/robots.txt`; don't fetch `Disallow`-ed paths; honour `Crawl-delay`. (§3) |
| **Read the Terms of Service** | Some sites *contractually* forbid scraping. Public ≠ free-to-take. |
| **Rate-limit yourself** | A human clicks every few seconds; don't hammer with 100 req/s. Add delays. (§4) |
| **Identify yourself** | A real `User-Agent` with contact info; never spoof to evade blocks. |
| **Don't take personal data** | Names, emails, faces → **GDPR/CCPA** territory. Don't collect PII without a lawful basis. |
| **Respect copyright** | Facts aren't copyrightable; creative content is. Don't republish wholesale. |
| **Cache, don't re-fetch** | Store what you pull so you never request the same page twice. (§4) |

> ⚖️ **Not legal advice.** Scraping law varies by jurisdiction and is evolving (e.g. *hiQ v. LinkedIn*). Public data scraped politely is generally lower-risk; bypassing auth, ignoring ToS, or collecting PII is higher-risk. When in doubt, ask, or use an official API.

## 2. The anatomy of a scrape

Every scraper, hand-rolled or managed, is the same four-step pipeline:

```text
   FETCH            PARSE              EXTRACT            STORE
   ─────            ─────              ───────            ─────
  HTTP GET   ──▶   HTML → tree   ──▶  pick the bits  ──▶  rows / JSON /
  (requests)       (BeautifulSoup)    you want           markdown → DB / RAG
```

The **fetch** you already know from Notebook 12. The new skill is **parse + extract**: turning a soup of tags into a tidy table. We'll do it for real against a local HTML fixture (a fictional bookshop page) so the cell runs with no network.

In [1]:
# A local HTML fixture — a stand-in for `requests.get(url).text`, so this runs offline & deterministically.
# (In real life: html = requests.get(url, headers=..., timeout=10).text  — see Notebook 12.)
HTML = '''
<html><head><title>The Polite Bookshop</title></head>
<body>
  <h1>Catalogue — page 1</h1>
  <section id="catalogue">
    <article class="product">
      <h3 class="title">Clean Code</h3>
      <p class="price">£32.50</p>
      <p class="rating" data-stars="4">★★★★</p>
      <span class="availability">In stock (12)</span>
      <a class="more" href="/catalogue/clean-code">details</a>
    </article>
    <article class="product">
      <h3 class="title">The Pragmatic Programmer</h3>
      <p class="price">£28.99</p>
      <p class="rating" data-stars="5">★★★★★</p>
      <span class="availability">In stock (3)</span>
      <a class="more" href="/catalogue/pragmatic-programmer">details</a>
    </article>
    <article class="product">
      <h3 class="title">Designing Data-Intensive Applications</h3>
      <p class="price">£41.00</p>
      <p class="rating" data-stars="5">★★★★★</p>
      <span class="availability">Out of stock</span>
      <a class="more" href="/catalogue/ddia">details</a>
    </article>
  </section>
  <a class="next" href="/catalogue/page-2">next →</a>
</body></html>
'''

In [2]:
# PARSE — turn the raw HTML string into a navigable tree (step 2 of fetch→parse→extract→store).
import re
import pandas as pd
from bs4 import BeautifulSoup

soup = BeautifulSoup(HTML, "html.parser")     # "html.parser" is built in; "lxml" is faster if installed

In [3]:
# EXTRACT + STORE — pull the fields we want from each product into a tidy DataFrame.
rows = []
for art in soup.select("article.product"):    # CSS selector: every <article class="product">
    rows.append({
        "title":     art.select_one("h3.title").get_text(strip=True),
        "price":     float(re.sub(r"[^0-9.]", "", art.select_one(".price").text)),
        "stars":     int(art.select_one(".rating")["data-stars"]),     # read an attribute
        "in_stock":  "In stock" in art.select_one(".availability").text,
        "url":       art.select_one("a.more")["href"],
    })

books = pd.DataFrame(rows)
print(books.to_string(index=False))
print(f"\n{len(books)} books · {books['in_stock'].sum()} in stock · avg £{books['price'].mean():.2f}")

                                title  price  stars  in_stock                             url
                           Clean Code  32.50      4      True           /catalogue/clean-code
             The Pragmatic Programmer  28.99      5      True /catalogue/pragmatic-programmer
Designing Data-Intensive Applications  41.00      5     False                 /catalogue/ddia

3 books · 2 in stock · avg £34.16


**That's the whole craft of parsing.** A few patterns cover ~90% of real pages:

| Goal | BeautifulSoup |
|---|---|
| First match | `soup.find("h3")` / `soup.select_one(".price")` |
| All matches | `soup.find_all("article")` / `soup.select("article.product")` |
| Text | `el.get_text(strip=True)` |
| Attribute | `el["href"]`, `el.get("data-stars")` |
| Nested | chain selectors: `art.select_one("a.more")["href"]` |

> ⚠️ **CSS selectors are brittle.** They're coupled to the site's *markup*, which changes without warning — a redesign that renames `.price` to `.product-price` silently breaks your scraper (you'll get `None` and an `AttributeError`). Defend yourself: prefer **stable hooks** (IDs, `data-*` attributes, semantic tags) over generated class names, guard every `.select_one()` with a `None` check, and **alert on zero rows** — "found 0 products" almost always means the layout moved, not that the shop is empty. This fragility is a big part of *why* the managed services in §6 exist.

---

### ✋ Quick exercise (~2 min) — Grab the heading and the next page

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A real catalogue spans many pages, so your scraper needs two things off page 1: a label for the data and where to go next. Using the existing `soup` from §2 (no re-fetching), pull the page's `<h1>` heading text and the `href` of the `next →` pager link (it has class `next`).

In [4]:
# ✍️ Your turn 👇
# Use the existing `soup` from §2 — no re-fetching.
heading  = ...   # the catalogue's <h1> text
next_url = ...    # the href of the "next →" pager link (class "next")
print(heading, "→ next page:", next_url)

Ellipsis → next page: Ellipsis


<details>
<summary>✅ <b>Solution</b></summary>

```python
heading  = soup.select_one("h1").get_text(strip=True)
next_url = soup.select_one("a.next")["href"]
print(heading, "→ next page:", next_url)
```

`select_one` grabs the first match; `.get_text(strip=True)` reads its text and `["href"]` reads an attribute — the exact patterns from the §2 table. Following `a.next` is how you'd page through the whole catalogue.
</details>

## 3. Respect `robots.txt`

`robots.txt` is a site's posted house rules for bots: which paths are open, which are off-limits, and how slowly to crawl. Python's standard library reads and enforces it for you — **no excuse to skip this check.**

In [5]:
# urllib.robotparser is standard library — it parses robots.txt and answers "may I fetch this?"
from urllib import robotparser

ROBOTS = '''
User-agent: *
Disallow: /cart/
Disallow: /checkout/
Disallow: /account/
Crawl-delay: 2
Allow: /

User-agent: GreedyBot
Disallow: /
'''.strip().splitlines()

rp = robotparser.RobotFileParser()
rp.parse(ROBOTS)                       # in production: rp.set_url(site + "/robots.txt"); rp.read()

UA = "PoliteCourseBot/1.0 (+https://example.com/bot-info)"
for path in ["/catalogue/page-2", "/cart/", "/account/settings", "/"]:
    allowed = rp.can_fetch(UA, "https://shop.example.com" + path)
    print(f"  {'✅ allowed ' if allowed else '🚫 blocked '} {path}")

print(f"\nCrawl-delay requested: {rp.crawl_delay(UA)} s  → sleep at least this long between requests")
print(f"GreedyBot may fetch '/': {rp.can_fetch('GreedyBot', 'https://shop.example.com/')}  (it's banned outright)")

  ✅ allowed  /catalogue/page-2
  🚫 blocked  /cart/
  🚫 blocked  /account/settings
  ✅ allowed  /

Crawl-delay requested: 2 s  → sleep at least this long between requests
GreedyBot may fetch '/': False  (it's banned outright)


## 4. Polite scraping — the four habits

A well-behaved scraper does four things every time. Three you already met in Notebook 12 (timeouts, retries, status checks); the new ones are **rate-limiting** and **caching**.

1. **Identify** — a real `User-Agent` with a contact URL.
2. **Throttle** — wait `crawl-delay` (or a sane default) between requests; never run unbounded parallel hits.
3. **Cache** — store every response so a re-run never re-fetches. Kind to the server, fast for you.
4. **Retry politely** — back off on `429`/`5xx` (Notebook 12's `fetch_with_retry`), and *stop* on a `429` that asks you to.

Here's a tiny `PoliteScraper` that wires up throttle + cache. It takes its `fetch` function as an argument so we can demo it offline; in production you'd pass one that calls `requests`.

In [6]:
# The polite fetcher itself — wires up THROTTLE + CACHE, with the network injected so it runs offline.
import time

class PoliteScraper:
    def __init__(self, fetch_fn, delay=0.3, user_agent=UA):
        self.fetch_fn = fetch_fn          # (url, headers) -> html ; inject requests in production
        self.delay = delay                # seconds between *network* hits (set from crawl_delay)
        self.user_agent = user_agent
        self._cache = {}                  # url -> html ; a real one would persist to disk/sqlite
        self._last = 0.0

    def get(self, url):
        if url in self._cache:            # CACHE: never fetch the same page twice
            print(f"  cache  {url}")
            return self._cache[url]
        wait = self.delay - (time.monotonic() - self._last)   # THROTTLE: respect the gap
        if wait > 0:
            time.sleep(wait)
        html = self.fetch_fn(url, {"User-Agent": self.user_agent})   # IDENTIFY via header
        self._last = time.monotonic()
        self._cache[url] = html
        print(f"  FETCH  {url}")
        return html

In [7]:
# Offline demo: a fake network that just hands back our fixture (no real HTTP).
def fake_network(url, headers):
    assert "User-Agent" in headers, "always identify yourself"
    return HTML

scraper = PoliteScraper(fake_network, delay=0.3)
t0 = time.monotonic()
scraper.get("https://shop.example.com/catalogue/page-1")   # FETCH (waits its turn)
scraper.get("https://shop.example.com/catalogue/page-2")   # FETCH (throttled by `delay`)
scraper.get("https://shop.example.com/catalogue/page-1")   # cache hit → instant, no network
print(f"\nelapsed {time.monotonic() - t0:.2f}s for 2 fetches + 1 cache hit (throttle respected)")

  FETCH  https://shop.example.com/catalogue/page-1


  FETCH  https://shop.example.com/catalogue/page-2
  cache  https://shop.example.com/catalogue/page-1

elapsed 0.31s for 2 fetches + 1 cache hit (throttle respected)


---

### ✋ Quick exercise (~2 min) — A robots-aware fetch loop

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A polite crawler checks the rules *before* it knocks. Loop over the two paths below and call `scraper.get(url)` **only** if `rp.can_fetch(UA, url)` returns `True` (reuse `rp` from §3 and `scraper`/`UA` from §4); print a skip message otherwise.

```python
paths = ["/catalogue/page-3", "/account/settings"]
```

In [8]:
# ✍️ Your turn 👇
# Reuse `rp` (§3), `scraper` and `UA` (§4) — no new objects needed.
for path in ["/catalogue/page-3", "/account/settings"]:
    url = "https://shop.example.com" + path
    # only call scraper.get(url) if robots.txt allows it; otherwise print a skip message
    ...

<details>
<summary>✅ <b>Solution</b></summary>

```python
for path in ["/catalogue/page-3", "/account/settings"]:
    url = "https://shop.example.com" + path
    if rp.can_fetch(UA, url):
        scraper.get(url)
    else:
        print(f"  🚫 skipped (robots.txt) {path}")
```

Checking `rp.can_fetch` *before* `scraper.get` means you never even queue a disallowed URL — politeness (§3) and the throttled/cached fetcher (§4) working together.
</details>

## 5. Where DIY scraping hurts

`requests` + BeautifulSoup is perfect for **static, well-structured HTML**. Modern sites are often neither. The pain points:

| Problem | Why `requests`+`bs4` struggles | The usual fix |
|---|---|---|
| **JavaScript rendering** | `requests` gets the *empty shell*; the content is drawn by JS in a browser | a **headless browser**: Playwright / Selenium |
| **Anti-bot / Cloudflare** | challenges, fingerprinting, IP blocks | rotating proxies, stealth browsers, or a managed service |
| **Layout drift** | selectors break on every redesign (§2 warning) | resilient selectors, monitoring — or LLM extraction |
| **Pagination / crawling** | you hand-write the link-following loop & dedup | a crawler that maps the whole site |
| **Many formats** | PDFs, infinite scroll, iframes each need bespoke code | one API that normalises them all |
| **"Just give me the text for my LLM"** | HTML is full of nav/ads/boilerplate noise | a service that returns clean **markdown** |

You *can* solve each by hand (Playwright for JS, proxies for blocks, custom crawlers…). But that's a lot of undifferentiated plumbing to build and babysit. Increasingly, the move is to hand the messy part to a **managed scraping API** — which is where **Firecrawl** comes in.

## 6. Firecrawl — the LLM-ready web API

**[Firecrawl](https://firecrawl.dev)** is a scraping service built for the AI era. You give it a URL; it handles the browser, the JavaScript, the anti-bot dance, and the boilerplate-stripping, and hands you back **clean markdown** (or **structured JSON**) — the format LLMs and RAG pipelines actually want. It exposes a handful of endpoints:

| Endpoint | Method (SDK) | What it does |
|---|---|---|
| **Scrape** | `scrape(url, formats=[...])` | one URL → markdown / html / links / screenshot / structured JSON |
| **Crawl** | `crawl(url, limit=...)` | follow links across a whole site → markdown for every page |
| **Map** | `map(url)` | fast — list *all* URLs on a site (great for planning a crawl) |
| **Search** | `search(query)` | web search **and** scrape the results in one call |
| **Extract / Agent** | `extract(...)` / `agent(...)` | LLM-powered: pull structured data matching your schema, even across pages |

> 🧠 **Why markdown is the point.** An LLM doesn't want `<div class="nav-wrapper">…</div>` — it wants the *content*. Firecrawl strips nav bars, ads, cookie banners, and scripts, and returns the page as the markdown a human would read. That output drops straight into a RAG chunker (Module 6, NB 23) or an agent's context (Module 8) with **no HTML cleanup code on your side**. "Scrape → markdown → embed" is the modern ingestion pipeline.

In [9]:
# Firecrawl SDK — the REAL usage is the commented block; the runnable part uses an offline mock
# so this notebook needs no API key or network.
#
#   from firecrawl import Firecrawl                 # pip install firecrawl-py
#   app = Firecrawl(api_key="fc-...")               # or set FIRECRAWL_API_KEY in the environment
#   doc = app.scrape("https://firecrawl.dev", formats=["markdown", "links"])
#   print(doc.markdown)                             # clean, LLM-ready markdown
#
# ---- Offline mock: mimics the v2 SDK surface (Firecrawl().scrape -> object with .markdown/.json) ----
class _Doc:
    def __init__(self, markdown=None, json=None, links=None):
        self.markdown, self.json, self.links = markdown, json, links or []

class MockFirecrawl:
    _MD = ("# The Polite Bookshop\n\n## Catalogue — page 1\n\n"
           "1. **Clean Code** — £32.50 (★★★★) — In stock\n"
           "2. **The Pragmatic Programmer** — £28.99 (★★★★★) — In stock\n"
           "3. **Designing Data-Intensive Applications** — £41.00 (★★★★★) — Out of stock\n")
    _BOOKS = [
        {"title": "Clean Code", "price": 32.50, "in_stock": True},
        {"title": "The Pragmatic Programmer", "price": 28.99, "in_stock": True},
        {"title": "Designing Data-Intensive Applications", "price": 41.00, "in_stock": False},
    ]
    def scrape(self, url, formats=None):
        formats = formats or ["markdown"]
        for f in formats:                                  # a {"type": "json", ...} format → structured data
            if isinstance(f, dict) and f.get("type") == "json":
                return _Doc(markdown=self._MD, json={"books": self._BOOKS})
        return _Doc(markdown=self._MD, links=[u + "/catalogue" for u in [url]])
    def crawl(self, url, limit=10, **kw):
        return {"status": "completed", "data": [_Doc(markdown=f"# Page {i}\n...") for i in range(min(3, limit))]}
    def map(self, url, limit=30, **kw):
        return {"links": [f"{url}/catalogue/page-{i}" for i in range(1, 4)]}

In [10]:
# Get a client: the real SDK if installed (+ API key), otherwise our offline mock.
try:
    from firecrawl import Firecrawl
    app, USING_REAL = Firecrawl(), True       # uses FIRECRAWL_API_KEY if present
except Exception:
    app, USING_REAL = MockFirecrawl(), False
print("client:", "REAL Firecrawl" if USING_REAL else "offline mock\n")

doc = app.scrape("https://shop.example.com", formats=["markdown"])
print(doc.markdown)

client: offline mock

# The Polite Bookshop

## Catalogue — page 1

1. **Clean Code** — £32.50 (★★★★) — In stock
2. **The Pragmatic Programmer** — £28.99 (★★★★★) — In stock
3. **Designing Data-Intensive Applications** — £41.00 (★★★★★) — Out of stock



### Structured extraction — markdown is nice, *typed JSON* is better

The real power is the **`json` format**: hand Firecrawl a schema (a Pydantic model) plus a natural-language prompt, and its LLM fills the schema from the page — no selectors, resilient to layout changes. This is the part DIY scraping can't easily match.

```python
# ── REAL Firecrawl structured extraction (reference) ──────────────────────
from pydantic import BaseModel, Field

class Book(BaseModel):
    title: str = Field(description="the book's title")
    price: float = Field(description="price in GBP")
    in_stock: bool

class Catalogue(BaseModel):
    books: list[Book]

doc = app.scrape(
    "https://shop.example.com",
    formats=[{"type": "json", "schema": Catalogue, "prompt": "Extract every book with price and stock."}],
)
print(doc.json)            # → {"books": [{"title": "...", "price": 32.5, "in_stock": true}, ...]}
```

```python
# ── Crawl a whole site, Map its URLs, or Search-and-scrape (reference) ────
site  = app.crawl("https://docs.example.com", limit=50)         # → markdown for up to 50 linked pages
urls  = app.map("https://docs.example.com")                     # → just the list of URLs, fast
hits  = app.search("best python forecasting library", limit=5)  # web search + scrape results
# async variant: from firecrawl import AsyncFirecrawl
```

In [11]:
# Run the structured-extraction flow on the mock, then VALIDATE it (Notebook 13's habit).
res = app.scrape(
    "https://shop.example.com",
    formats=[{"type": "json", "prompt": "Extract every book with price and stock.", "schema": None}],
)
raw = res.json["books"]
print("raw JSON from Firecrawl:", raw[0], "...\n")

raw JSON from Firecrawl: {'title': 'Clean Code', 'price': 32.5, 'in_stock': True} ...



In [12]:
# Validate at the boundary. Use Pydantic if installed (the production choice), else a stdlib dataclass.
try:
    from pydantic import BaseModel
    class Book(BaseModel):
        title: str
        price: float
        in_stock: bool
    books_valid = [Book(**b) for b in raw]
    engine = "pydantic"
except ImportError:
    from dataclasses import dataclass
    @dataclass
    class Book:
        title: str
        price: float
        in_stock: bool
    books_valid = [Book(**b) for b in raw]
    engine = "dataclass (pydantic not installed)"

In [13]:
# Tabulate the validated objects → a DataFrame ready to embed (NB 18) or load into SQL (NB 13).
df = pd.DataFrame([vars(b) if not hasattr(b, "model_dump") else b.model_dump() for b in books_valid])
print(f"validated {len(df)} rows with {engine}:")
print(df.to_string(index=False))
print("\n→ This DataFrame is now ready to embed (Module 5, NB 18) or load into SQL (NB 13).")

validated 3 rows with dataclass (pydantic not installed):
                                title  price  in_stock
                           Clean Code  32.50      True
             The Pragmatic Programmer  28.99      True
Designing Data-Intensive Applications  41.00     False

→ This DataFrame is now ready to embed (Module 5, NB 18) or load into SQL (NB 13).


---

### ✋ Quick exercise (~2 min) — Summarize the ingested catalogue

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Right after ingestion you usually log a quick health check on what came in. Using the validated `df` from the cell above (no re-scraping), filter to the **in-stock** books and print how many there are plus their **average price**.

In [14]:
# ✍️ Your turn 👇
# Reuse the validated `df` from the cell above — no re-scraping.
# Keep only the in-stock books, then print how many and their average price.
...

Ellipsis

<details>
<summary>✅ <b>Solution</b></summary>

```python
in_stock = df[df["in_stock"]]
print(f"{len(in_stock)} in stock · avg £{in_stock['price'].mean():.2f}")
```

`df[df["in_stock"]]` is a boolean mask that keeps only available titles before averaging — a one-line ingestion summary you'd log after every scrape.
</details>

## 7. Choosing your approach

```text
Is there an official API / data export / RSS?           →  use it (Notebook 12). Stop here.
Static HTML, you control the selectors, low volume?     →  requests + BeautifulSoup  (§2–§4)
Need JavaScript rendered, but staying in-house?         →  Playwright / Selenium (headless browser)
JS + anti-bot + many pages + "just give me markdown"?   →  Firecrawl (or a managed scraping API)
Need typed records out of messy pages, no selectors?    →  Firecrawl  json  extraction (LLM)
Feeding a RAG index or an agent?                        →  Firecrawl markdown → chunk → embed
```

> 🧱 **Where this sits in an AI system.** Scraping is an **ingestion** step. The clean markdown / typed JSON you produce here is the *input* to the rest of the course: chunk + embed it for **retrieval** (Module 6, NB 23), validate it with **Pydantic** (NB 13), give it to an **agent** as a tool (Module 8), or schedule the whole thing as a recurring **job** (NB 40). A scraper is rarely the product — it's the pipe that fills the product.

## 🧪 Exercises

### Exercise 1 — Extract the navigation links
Using `soup` from §2, collect **every** link on the page as `(text, href)` pairs — including the "details" links *and* the "next →" pager. (Hint: `soup.find_all("a")`, then read `.get_text()` and `["href"]`.)

In [15]:
# ── Exercise 1 solution ──────────────────────────────────────────────────
links = [(a.get_text(strip=True), a["href"]) for a in soup.find_all("a", href=True)]
for text, href in links:
    print(f"  {text:<10} → {href}")
print(f"\n{len(links)} links found (next-page link lets you follow pagination — see Notebook 12 §8).")

  details    → /catalogue/clean-code
  details    → /catalogue/pragmatic-programmer
  details    → /catalogue/ddia
  next →     → /catalogue/page-2

4 links found (next-page link lets you follow pagination — see Notebook 12 §8).


### Exercise 2 — Design an extraction schema
You want to scrape conference talks into a database. Define the **schema** you'd hand Firecrawl's `json` format for a talk with: title, speaker, track, and start time — then run it against the mock to see the typed output. (Reuse the try/except Pydantic-or-dataclass pattern from §6.)

In [16]:
# ── Exercise 2 solution ──────────────────────────────────────────────────
# The schema you'd pass: formats=[{"type": "json", "schema": Talk, "prompt": "Extract all talks."}]
try:
    from pydantic import BaseModel
    class Talk(BaseModel):
        title: str
        speaker: str
        track: str
        start: str        # ISO time string; use datetime in production
except ImportError:
    from dataclasses import dataclass
    @dataclass
    class Talk:
        title: str
        speaker: str
        track: str
        start: str

# Mock what Firecrawl's LLM extraction would return for this schema:
extracted = [
    {"title": "Conformal Prediction in Practice", "speaker": "A. Researcher", "track": "ML", "start": "2026-09-01T09:00"},
    {"title": "Scraping the Modern Web",          "speaker": "B. Engineer",   "track": "Data", "start": "2026-09-01T10:30"},
]
talks = [Talk(**t) for t in extracted]
print(f"{len(talks)} talks validated against the schema:")
for t in talks:
    print(f"  {t.start}  [{t.track}]  {t.title} — {t.speaker}")

2 talks validated against the schema:
  2026-09-01T09:00  [ML]  Conformal Prediction in Practice — A. Researcher
  2026-09-01T10:30  [Data]  Scraping the Modern Web — B. Engineer


## 🧠 Key takeaways

- **Scrape only as a last resort** — API > export/RSS > scraping — and always **within the rules**: `robots.txt`, ToS, rate limits, no PII, respect copyright.
- The DIY craft is **fetch → parse → extract → store**: `requests` for the fetch, **BeautifulSoup** selectors for the parse. Make the fetcher **polite** (identify, throttle, cache, retry).
- **Selectors are brittle.** Prefer stable hooks, guard for `None`, and alert on zero rows — a redesign breaks scrapers silently.
- DIY breaks down on **JavaScript, anti-bot, crawling, and "give me clean text."** That's the niche for **Firecrawl** (or a headless browser for in-house JS).
- **Firecrawl** turns any URL into **LLM-ready markdown** or **schema-validated JSON** in one call — `scrape` / `crawl` / `map` / `search` / `extract`. The markdown output is built to flow straight into RAG and agents.
- Scraping is **ingestion**, not the product: validate the output (Pydantic, NB 13) and feed it to retrieval, agents, or a scheduled job.

## 🚀 Next step

Take the markdown this notebook produces and chunk + embed it in **`../06_ai_engineering/23_embeddings_retrieval.ipynb`**, or wire a "fetch this URL" Firecrawl call as a **tool** for the agent in **`../08_agents_tools_mcp/32_designing_robust_tools.ipynb`**.